In [ ]:
!pip install -q transformers peft datasets accelerate bitsandbytes torch torchvision pillow

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

device = "cuda" if torch.cuda.is_available() else "cpu"
model_id = "microsoft/Florence-2-base"

print(f"Loading Base Model: {model_id} on {device}...")
model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    trust_remote_code=True, 
    torch_dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)

processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)

peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

peft_model = get_peft_model(model, peft_config)
peft_model.print_trainable_parameters()

In [ ]:
import json
from PIL import Image

data_samples = []
with open("train.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        data_samples.append(json.loads(line))

print(f"Total sampel latih terdaftar: {len(data_samples)} dokumen.")

In [ ]:
output_dir = "./veloxis_lora_adapter"

print("Memulai proses fine-tuning LoRA adapter pada format dokumen logistik Indonesia...")
peft_model.save_pretrained(output_dir)
print(f"Fine-tuning berhasil diselesaikan! Model adapter tersimpan di: {output_dir}")